# Prairie Vigie – notebook 1 : RPG 2015-2023, mesure des conversions présumées, premier modèle
**Données** : RPG parcelles graphiques 2015-2023 (IGN, Licence Ouverte) ; contours communaux (API Géo, Etalab).

### Règle de décision
1. **Mesure fiable** : téléchargement complet (moins de 0,5 % d'objets manquants par département et par année)
   ET prairies permanentes 2018 de la région à moins de 10 % du chiffre officiel (673 869 ha).
2. **Signal prédictif (horizon 1 an)** : modèle régional, chaque département laissé de côté à son tour,
   test 2022→2023 : lift ≥ 2 sur les 10 % les plus à risque **dans chacun des 5 départements**,
   ET robustesse : lift régional ≥ 2 sur 2021→2022.
3. **Usage opérationnel (horizon 2 ans)** : le RPG de l'année N n'est publié qu'en N+1, donc on prédit
   l'année N+2. Si le lift régional (départements laissés de côté) est ≥ 2, la carte est présentée comme
   une **carte de risque** ; sinon comme une **carte de vigilance descriptive**.
4. **Prairies anciennes** (jamais cultivées depuis 2015) : résultats affichés, sans seuil bloquant.

**Vocabulaire** : une *conversion présumée* est une prairie permanente dont plus de la moitié est déclarée en terre arable
l'année suivante (déclarations PAC, sans constat de terrain). Les colonnes `n_retournees`, `ha_retournees`, `ha_retournes`
gardent leur nom historique : elles comptent des conversions présumées.

In [ ]:
# 0.Sur Google Colab, le dossier DOSSIER est créé (ou repris) dans « Mon Drive »
#    Les 4 notebooks doivent utiliser le même dossier.
DOSSIER = "datathon_prairies"
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(os.path.join("/content/drive/MyDrive", DOSSIER), exist_ok=True)
    os.chdir(os.path.join("/content/drive/MyDrive", DOSSIER))
except ImportError:
    pass
print("Dossier de travail :", os.getcwd())

In [ ]:
# 0 bis. Bibliothèques
!pip -q install geopandas pyogrio pyarrow scikit-learn

In [ ]:
# 1. Paramètres
import json, time, gc, xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, requests
from shapely.geometry import shape, box
from shapely.validation import make_valid
from shapely.ops import unary_union

DEPS = ["14", "27", "50", "61", "76"]
HISTO = [2015, 2016, 2017, 2018]            # sert seulement à savoir si une parcelle a été cultivée
ANNEES_PP = [2019, 2020, 2021, 2022, 2023]   # définition « prairie permanente » stable à partir de 2019
ANNEES = HISTO + ANNEES_PP
TRANSITIONS = [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023)]
TEST, ROBUSTESSE = (2022, 2023), (2021, 2022)
A0_SCORE = 2023                              # année de départ de la carte de risque (horizon 2 ans -> 2025)
TAMPON = 5000                                # 5 km autour du département pour les variables de voisinage
WFS = "https://data.geopf.fr/wfs/ows"
PAGE, DALLE = 5000, 20_000
CACHE = Path("cache_rpg"); CACHE.mkdir(exist_ok=True)
SORTIES = Path("sorties_prairies"); SORTIES.mkdir(exist_ok=True)
SESSION = requests.Session()
CHAMPS = {"code_cultu", "code_group"}       # liste blanche : aucun identifiant d'exploitant

PP, HERBE, GEL = 18, {17, 18, 19}, {11, 12, 13}
ARABLE = {1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 16, 24, 25, 26}
def classe(g):
    g = int(g) if pd.notna(g) else -1
    return ("pp" if g == PP else "herbe" if g in HERBE else "arable" if g in ARABLE
            else "gel" if g in GEL else "autre")

def polygone(g):
    """Géométrie valide, parties surfaciques seulement."""
    g = make_valid(g)
    if g.geom_type == "GeometryCollection":
        parts = [p for p in g.geoms if p.geom_type in ("Polygon", "MultiPolygon")]
        g = unary_union(parts) if parts else None
    return g

In [ ]:
# 2. Communes, couches RPG, test du tri côté serveur
def communes_dep(dep):
    f = CACHE / f"communes_{dep}.parquet"
    if f.exists():
        return gpd.read_parquet(f)
    r = SESSION.get(f"https://geo.api.gouv.fr/departements/{dep}/communes",
                    params={"geometry": "contour", "format": "geojson", "fields": "code,nom,codeEpci"}, timeout=180)
    r.raise_for_status()
    g = gpd.GeoDataFrame.from_features(r.json()["features"], crs=4326).to_crs(2154)
    g.to_parquet(f)
    return g

COMMUNES = gpd.GeoDataFrame(pd.concat([communes_dep(d).assign(dep=d) for d in DEPS], ignore_index=True),
                            geometry="geometry", crs=2154)
if "codeEpci" not in COMMUNES:
    COMMUNES["codeEpci"] = None
CONTOURS = {d: make_valid(unary_union(COMMUNES.loc[COMMUNES.dep == d, "geometry"].values)) for d in DEPS}
for d in DEPS:
    print(f"Département {d} : {(COMMUNES.dep == d).sum()} communes, {CONTOURS[d].area / 1e4:,.0f} ha")

r = SESSION.get(WFS, params={"SERVICE": "WFS", "REQUEST": "GetCapabilities", "VERSION": "2.0.0"}, timeout=180)
r.raise_for_status()
NOMS = {el.text for el in ET.fromstring(r.content).iter() if el.tag.endswith("}Name") and el.text}
COUCHES = {a: f"RPG.{a}:parcelles_graphiques" for a in ANNEES if f"RPG.{a}:parcelles_graphiques" in NOMS}
print("Couches utilisées :", json.dumps(COUCHES, indent=1))
ANNEES = [a for a in ANNEES if a in COUCHES]
HISTO = [a for a in HISTO if a in COUCHES]

# Tri côté serveur : rend la pagination stable (sans tri, des objets peuvent être sautés entre deux pages)
try:
    t = SESSION.get(WFS, params={"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
                                 "TYPENAMES": COUCHES[2019], "OUTPUTFORMAT": "application/json",
                                 "COUNT": 2, "SORTBY": "id_parcel ASC"}, timeout=120)
    TRI = t.status_code == 200 and "features" in t.json()
except Exception:
    TRI = False
print("Tri côté serveur disponible :", TRI)

In [ ]:
# 3. Téléchargement par dalles (département + 5 km), trié, filtré, dédoublonné, contrôlé, en cache
def get(params):
    for essai in range(6):
        try:
            r = SESSION.get(WFS, params=params, timeout=300)
            if r.status_code == 200:
                return r.json()
        except (requests.RequestException, ValueError):
            pass
        time.sleep(5 * 2 ** essai)
    raise RuntimeError(f"WFS indisponible : {params.get('TYPENAMES')}")

def charger(dep, annee):
    fichier = CACHE / f"rpg_{dep}_{annee}_v3.parquet"
    if fichier.exists():
        return gpd.read_parquet(fichier)
    zone = CONTOURS[dep].buffer(TAMPON)
    xmin, ymin, xmax, ymax = zone.bounds
    vus, lignes, champs_bruts = set(), [], set()
    annonces, recus = 0, 0                     # contrôle d'exhaustivité
    for x in np.arange(xmin, xmax, DALLE):
        for y in np.arange(ymin, ymax, DALLE):
            if not box(x, y, x + DALLE, y + DALLE).intersects(zone):
                continue
            debut, ids_dalle, annonce = 0, set(), None
            while True:
                params = {"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
                          "TYPENAMES": COUCHES[annee], "SRSNAME": "EPSG:2154",
                          "BBOX": f"{x},{y},{x + DALLE},{y + DALLE},EPSG:2154",
                          "OUTPUTFORMAT": "application/json", "COUNT": PAGE, "STARTINDEX": debut}
                if TRI:
                    params["SORTBY"] = "id_parcel ASC"
                page = get(params)
                m = page.get("numberMatched", page.get("totalFeatures"))
                if isinstance(m, int):
                    annonce = m
                feats = page.get("features", [])
                for f in feats:
                    cle = f.get("id") or json.dumps(f.get("geometry"))[:200]
                    ids_dalle.add(cle)
                    if cle in vus or not f.get("geometry"):
                        continue
                    vus.add(cle)
                    props = {k.lower(): v for k, v in (f.get("properties") or {}).items()}
                    champs_bruts |= set(props)
                    lignes.append({**{k: props.get(k) for k in CHAMPS}, "geometry": shape(f["geometry"])})
                if len(feats) < PAGE:
                    break
                debut += PAGE
            if annonce is not None:
                annonces += annonce
                recus += len(ids_dalle)
    manque = (annonces - recus) / annonces if annonces else float("nan")
    (CACHE / f"complet_{dep}_{annee}.json").write_text(json.dumps({"annonces": annonces, "recus": recus}))
    # On affiche seulement le nombre de champs (ni leurs noms ni leurs valeurs)
    print(f"    {dep} {annee} : {len(champs_bruts)} champs dans la source, {len(CHAMPS)} conservés ; objets manquants = {manque:.2%}")
    g = gpd.GeoDataFrame(lignes, geometry="geometry", crs=2154)
    g["geometry"] = g.geometry.apply(polygone)
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    g = g[g.representative_point().within(zone)].copy()
    g["dans_dep"] = g.representative_point().within(CONTOURS[dep])   # seules ces parcelles sont comptées
    g["code_group"] = pd.to_numeric(g["code_group"], errors="coerce")
    g = g.reset_index(drop=True)
    g.to_parquet(fichier)
    return g

for d in DEPS:
    for a in ANNEES:
        t0 = time.time()
        n = len(charger(d, a))
        print(f"{d} {a} : {n:,} parcelles ({time.time() - t0:.0f} s)")

In [ ]:
# 4. Contrôles de la mesure : exhaustivité, surfaces, chiffre officiel 2018
comp = []
for d in DEPS:
    for a in ANNEES:
        c = json.loads((CACHE / f"complet_{d}_{a}.json").read_text())
        comp.append({"dep": d, "annee": a, **c,
                     "manque": (c["annonces"] - c["recus"]) / c["annonces"] if c["annonces"] else np.nan})
COMP = pd.DataFrame(comp)
print("Objets manquants (part) par département et année :")
print(COMP.pivot(index="annee", columns="dep", values="manque").round(4).to_string())
if (COMP["annonces"] == 0).all():
    print("⚠ Le serveur n'annonce pas le nombre d'objets : exhaustivité NON vérifiable (à signaler).")
CRIT_COMPLET = bool((COMP["manque"].fillna(0) <= 0.005).all())

lignes = []
for d in DEPS:
    for a in ANNEES:
        g = charger(d, a)
        g = g[g["dans_dep"]]
        s = (g.area / 1e4).groupby(g["code_group"].apply(classe)).sum()
        lignes.append({"dep": d, "annee": a, "n_parcelles": len(g), **s.round(0).to_dict()})
SURF = pd.DataFrame(lignes).fillna(0)
SURF.to_csv(SORTIES / "surfaces_par_dep_annee.csv", index=False)
reg = SURF.groupby("annee")[[c for c in ["n_parcelles", "pp", "herbe", "arable", "gel", "autre"] if c in SURF]].sum()
reg["variation_pp"] = reg["pp"].pct_change().round(3)
print("\nNormandie, surfaces en ha :")
print(reg.to_string())
OFFICIEL_2018 = 673_869
if 2018 in reg.index:
    ecart = reg.loc[2018, "pp"] / OFFICIEL_2018 - 1
    CRIT_OFFICIEL = bool(abs(ecart) <= 0.10)
    print(f"\nPP 2018 RPG = {reg.loc[2018, 'pp']:,.0f} ha ; officiel = {OFFICIEL_2018:,} ha ; écart {ecart:+.1%}")
else:
    CRIT_OFFICIEL = False
    print("2018 indisponible : contrôle officiel impossible")
CRIT_MESURE = CRIT_COMPLET and CRIT_OFFICIEL
print("Exhaustivité :", "OK" if CRIT_COMPLET else "ÉCHEC", "| Chiffre officiel :", "OK" if CRIT_OFFICIEL else "ÉCHEC")

In [ ]:
# 5. Par département : devenir des prairies, gains, observatoire communal, variables, étiquettes à 2 ans
def devenir(D, a0, a1):
    g0 = D[a0]
    pp = g0.loc[(g0["classe"] == "pp") & g0["dans_dep"], ["pid", "geometry"]].copy()
    pp["surf0"] = pp.area
    inter = gpd.overlay(pp[["pid", "geometry"]], D[a1][["classe", "geometry"]], how="intersection", keep_geom_type=True)
    inter["a"] = inter.area
    t = inter.pivot_table(index="pid", columns="classe", values="a", aggfunc="sum", fill_value=0)
    t = pp.set_index("pid")[["surf0"]].join(t).fillna(0)
    cl = ["pp", "herbe", "arable", "gel", "autre"]
    for c in cl:
        t[f"f_{c}"] = (t[c] / t["surf0"]).clip(0, 1) if c in t else 0.0
    t["f_absent"] = (1 - t[[f"f_{c}" for c in cl]].sum(axis=1)).clip(0, 1)
    t["devenir"] = t[[f"f_{c}" for c in cl + ["absent"]]].idxmax(axis=1).str[2:]
    t["y"] = (t["f_arable"] > 0.5).astype(int)             # conversion présumée : plus de la moitié déclarée en terre arable
    return t[["surf0", "devenir", "y"]].reset_index()

def gains(D, a0, a1):
    """Prairies permanentes de a1 dont plus de la moitié était en terre arable en a0."""
    g1 = D[a1]
    pp1 = g1.loc[(g1["classe"] == "pp") & g1["dans_dep"], ["pid", "geometry"]].copy()
    pp1["s"] = pp1.area
    ar0 = D[a0].loc[D[a0]["classe"] == "arable", ["geometry"]]
    inter = gpd.overlay(pp1[["pid", "geometry"]], ar0, how="intersection", keep_geom_type=True)
    inter["a"] = inter.area
    f = inter.groupby("pid")["a"].sum() / pp1.set_index("pid")["s"]
    return pp1[pp1["pid"].isin(f[f > 0.5].index)]

def points(geoms):
    return gpd.GeoDataFrame(geometry=gpd.GeoSeries(geoms, crs=2154).representative_point().values, crs=2154)

def au_point(pts, g, cols):
    """Attributs de la parcelle de g située sous chaque point (première trouvée)."""
    j = gpd.sjoin(pts, g[cols + ["geometry"]], how="left", predicate="within")
    return j[~j.index.duplicated()].reindex(pts.index)[cols]

def stats_maille(g, taille):
    p = g.representative_point()
    df = pd.DataFrame({"cx": (p.x // taille).astype(int), "cy": (p.y // taille).astype(int),
                       "ha": g.area / 1e4, "classe": g["classe"]})
    s = df.pivot_table(index=["cx", "cy"], columns="classe", values="ha", aggfunc="sum", fill_value=0)
    tot = s.sum(axis=1)
    return pd.DataFrame({"part_arable": s.get("arable", 0) / tot, "part_pp": s.get("pp", 0) / tot})

def commune_de(geoms, dep):
    c = COMMUNES[COMMUNES.dep == dep][["code", "nom", "codeEpci", "geometry"]]
    return au_point(points(geoms), c, ["code", "nom", "codeEpci"])

def variables(D, LAB, a0, a1=None, dep=None):
    """Variables connues l'année a0 pour chaque prairie permanente du département (aucune fuite du futur)."""
    g = D[a0]
    pp = g[(g["classe"] == "pp") & g["dans_dep"]].reset_index(drop=True)
    p = points(pp.geometry.values)
    X = pd.DataFrame({"pid": pp["pid"].values})
    X["surface_ha"] = pp.area.values / 1e4
    X["compacite"] = (4 * np.pi * pp.area / pp.length ** 2).values
    X["code_cultu"] = pp["code_cultu"].astype(str).values
    h = pd.DataFrame({a: au_point(p, D[a], ["classe"])["classe"].fillna("absent").values
                      for a in ANNEES if a < a0})
    cultivee = h.isin(["arable", "gel"])
    X["n_annees_cultivee"] = cultivee.sum(axis=1).values
    X["deja_cultivee"] = cultivee.any(axis=1).astype(int).values
    cols_pp = [a for a in h.columns if a in ANNEES_PP]
    X["deja_pt"] = (h[cols_pp] == "herbe").any(axis=1).astype(int).values if cols_pp else 0
    toujours_pp = (h[cols_pp] == "pp").all(axis=1) if cols_pp else pd.Series(True, index=h.index)
    X["ancienne"] = (~cultivee.any(axis=1) & toujours_pp).astype(int).values   # jamais cultivée depuis 2015
    X["cx1"] = (p.geometry.x // 1000).astype(int).values
    X["cy1"] = (p.geometry.y // 1000).astype(int).values
    for taille, nom in [(1000, "1km"), (5000, "5km")]:
        idx = pd.MultiIndex.from_arrays([(p.geometry.x // taille).astype(int).values,
                                         (p.geometry.y // taille).astype(int).values])
        s = stats_maille(g, taille).reindex(idx)
        X[f"part_arable_{nom}"], X[f"part_pp_{nom}"] = s["part_arable"].values, s["part_pp"].values
        prev = [a for a in ANNEES if a < a0]
        if prev:
            s0 = stats_maille(D[prev[-1]], taille).reindex(idx)
            X[f"delta_arable_{nom}"] = (s["part_arable"].values - s0["part_arable"].values)
        tp = next((t for t in TRANSITIONS if t[1] == a0), None)
        if tp:
            d = LAB[tp].merge(D[tp[0]][["pid", "geometry"]], on="pid")
            q = gpd.GeoSeries(d["geometry"], crs=2154).representative_point()
            rr = pd.DataFrame({"cx": (q.x // taille).astype(int), "cy": (q.y // taille).astype(int),
                               "y": d["y"].values}).groupby(["cx", "cy"])["y"].mean()
            X[f"taux_ret_prec_{nom}"] = rr.reindex(idx).values
    X["commune"] = commune_de(pp.geometry.values, dep)["code"].values
    if a1 is not None:
        X = X.merge(LAB[(a0, a1)][["pid", "y", "devenir"]], on="pid", how="left")
        X["transition"] = f"{a0}->{a1}"
    X["a0"] = a0
    return X

def etiquette_2ans(D, LAB, a0):
    """Horizon 2 ans : prairie de a0 encore prairie en a0+1, puis convertie entre a0+1 et a0+2."""
    if (a0 + 1, a0 + 2) not in LAB:
        return None
    g = D[a0]
    pp = g[(g["classe"] == "pp") & g["dans_dep"]].reset_index(drop=True)
    suiv = D[a0 + 1][["pid", "classe", "geometry"]].rename(columns={"pid": "pid1", "classe": "classe1"})
    a = au_point(points(pp.geometry.values), suiv, ["pid1", "classe1"])
    a["pid"] = pp["pid"].values
    a = a[a["classe1"] == "pp"]
    lab = LAB[(a0 + 1, a0 + 2)][["pid", "y"]].rename(columns={"pid": "pid1", "y": "y2"})
    return a.merge(lab, on="pid1", how="left").fillna({"y2": 0})[["pid", "y2"]]

for dep in DEPS:
    fin = SORTIES / f"variables_{dep}.parquet"
    if fin.exists():
        print(f"{dep} : déjà calculé")
        continue
    t0 = time.time()
    D = {}
    for a in ANNEES:
        D[a] = charger(dep, a)
        D[a]["pid"] = np.arange(len(D[a]))
        D[a]["classe"] = D[a]["code_group"].apply(classe)
    LAB, BILAN, OBS, tabs = {}, [], [], []          # *_retournees = conversions présumées
    for a0, a1 in TRANSITIONS:
        LAB[(a0, a1)] = lab = devenir(D, a0, a1)
        gsel = gains(D, a0, a1)
        BILAN.append({"dep": dep, "transition": f"{a0}->{a1}", "n_pp": len(lab), "ha_pp": round(lab.surf0.sum() / 1e4),
                      "n_retournees": int(lab.y.sum()), "ha_retournees": round(lab.loc[lab.y == 1, "surf0"].sum() / 1e4),
                      "n_sorties_rpg": int((lab.devenir == "absent").sum()),
                      "n_gains": len(gsel), "ha_gains": round(gsel["s"].sum() / 1e4)})
        ret = lab[lab.y == 1].merge(D[a0][["pid", "geometry"]], on="pid")
        OBS.append(pd.concat([commune_de(ret["geometry"].values, dep).assign(ha=ret["surf0"].values / 1e4, type="ha_retournes"),
                              commune_de(gsel["geometry"].values, dep).assign(ha=gsel["s"].values / 1e4, type="ha_crees")])
                   .assign(dep=dep, transition=f"{a0}->{a1}"))
    for a0, a1 in TRANSITIONS:
        X = variables(D, LAB, a0, a1, dep)
        e2 = etiquette_2ans(D, LAB, a0)
        X = X.merge(e2, on="pid", how="left") if e2 is not None else X.assign(y2=np.nan)
        X.loc[X["y"] == 1, "y2"] = np.nan                    # déjà convertie à 1 an : hors étiquette 2 ans
        tabs.append(X)
    pd.concat(tabs, ignore_index=True).assign(dep=dep).to_parquet(SORTIES / f"variables_{dep}_tmp.parquet")
    variables(D, LAB, A0_SCORE, None, dep).assign(dep=dep).to_parquet(SORTIES / f"score_{dep}_{A0_SCORE}.parquet")
    pd.DataFrame(BILAN).to_csv(SORTIES / f"bilan_{dep}.csv", index=False)
    pd.concat(OBS).groupby(["dep", "code", "nom", "codeEpci", "transition", "type"], dropna=False)["ha"].sum() \
      .unstack("type", fill_value=0).round(1).reset_index().to_csv(SORTIES / f"observatoire_communes_{dep}.csv", index=False)
    (SORTIES / f"variables_{dep}_tmp.parquet").rename(fin)  # département terminé
    print(f"{dep} terminé ({time.time() - t0:.0f} s)")
    del D, LAB; gc.collect()

BILAN_DF = pd.concat([pd.read_csv(SORTIES / f"bilan_{d}.csv", dtype={"dep": str}) for d in DEPS])
BILAN_DF.to_csv(SORTIES / "bilan_transitions_normandie.csv", index=False)
pd.concat([pd.read_csv(SORTIES / f"observatoire_communes_{d}.csv", dtype={"dep": str, "code": str, "codeEpci": str})
           for d in DEPS]).to_csv(SORTIES / "observatoire_communes.csv", index=False)
print(BILAN_DF.to_string(index=False))
print("\nNormandie :")
print(BILAN_DF.groupby("transition")[["n_pp", "ha_pp", "n_retournees", "ha_retournees", "n_gains", "ha_gains",
                                     "n_sorties_rpg"]].sum().to_string())

In [ ]:
 # 6. Modèles : départements laissés de côté, horizon 1 an et 2 ans, intervalles de confiance
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

TAB = pd.concat([pd.read_parquet(SORTIES / f"variables_{d}.parquet") for d in DEPS], ignore_index=True)
CODES = list(TAB["code_cultu"].value_counts().index[:8])
def codes_onehot(df):
    for c in CODES:
        df[f"code_{c}"] = (df["code_cultu"] == c).astype(int)
    return df
TAB = codes_onehot(TAB)
NON_VARS = {"pid", "y", "y2", "devenir", "transition", "code_cultu", "dep", "a0", "cx1", "cy1", "commune"}
VARS = [c for c in TAB.columns if c not in NON_VARS]
print("Variables :", VARS)
print(TAB.groupby(["dep", "transition"])[["y", "y2", "ancienne"]].mean().round(4).to_string())

RNG = np.random.default_rng(0)
def lift10(y, s):
    k = max(1, int(0.1 * len(y)))
    return y[np.argsort(-s)[:k]].mean() / y.mean() if y.mean() > 0 else np.nan

def evaluer(y, s, nboot=200):
    y, s = np.asarray(y, float), np.nan_to_num(np.asarray(s, float))
    if y.sum() == 0:
        return dict(AUC_PR=np.nan, lift=np.nan, lift_bas=np.nan, lift_haut=np.nan)
    boots = []
    for _ in range(nboot):
        i = RNG.integers(0, len(y), len(y))
        boots.append(lift10(y[i], s[i]))
    return dict(AUC_PR=round(average_precision_score(y, s), 4), lift=round(lift10(y, s), 2),
                lift_bas=round(np.nanpercentile(boots, 2.5), 2), lift_haut=round(np.nanpercentile(boots, 97.5), 2))

def modele():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)

def ajuster_predire(tr, te, cible):
    """Entraîne sur les variables renseignées dans l'entraînement (une colonne vide fait planter le modèle)."""
    cols = [v for v in VARS if tr[v].notna().any()]
    m = modele().fit(tr[cols], tr[cible])
    return m.predict_proba(te[cols])[:, 1], m, cols

def campagne(cible, a0_test, nom_test):
    """Entraîne sur les années antérieures, évalue chaque département laissé de côté + modèle propre."""
    lignes = []
    base = TAB[TAB[cible].notna()]
    for dep in DEPS:
        te = base[(base.dep == dep) & (base.a0 == a0_test)]
        if te.empty:
            continue
        tr_reg = base[(base.dep != dep) & (base.a0 < a0_test - (1 if cible == "y2" else 0))]
        tr_dep = base[(base.dep == dep) & (base.a0 < a0_test - (1 if cible == "y2" else 0))]
        anc = te["ancienne"].values == 1
        for nom, tr in [("régional (dép. exclu)", tr_reg), ("propre au département", tr_dep), ("référence", None)]:
            s = (te["part_arable_1km"].values if tr is None
                 else ajuster_predire(tr, te, cible)[0])
            r_all, r_anc = evaluer(te[cible], s), evaluer(te[cible].values[anc], s[anc])
            lignes.append({"test": nom_test, "dep": dep, "modele": nom, "n": len(te), "positifs": int(te[cible].sum()),
                           "taux": round(te[cible].mean(), 4), **r_all,
                           "positifs_anciennes": int(te[cible].values[anc].sum()), "lift_anciennes": r_anc["lift"],
                           "lift_anciennes_bas": r_anc["lift_bas"], "lift_anciennes_haut": r_anc["lift_haut"]})
    return lignes

RES = pd.DataFrame(campagne("y", TEST[0], "1 an, 2022->2023")
                   + campagne("y", ROBUSTESSE[0], "1 an, 2021->2022 (robustesse)")
                   + campagne("y2", 2021, "2 ans, 2021 -> conversion 2022->2023"))
# Lift régional global (tous départements réunis, modèles « département exclu »)
glob = []
for test, g in RES[RES.modele == "régional (dép. exclu)"].groupby("test"):
    glob.append({"test": test, "lift_moyen_pondere": round(np.average(g["lift"], weights=g["positifs"]), 2)})
GLOB = pd.DataFrame(glob)
pd.set_option("display.width", 250)
print(RES.to_string(index=False))
print("\n", GLOB.to_string(index=False))
RES.to_csv(SORTIES / "resultats_modeles_normandie.csv", index=False)

In [ ]:
# 7. Décision (règle écrite avant les résultats)
r1 = RES[(RES.test == "1 an, 2022->2023") & (RES.modele == "régional (dép. exclu)")].set_index("dep")["lift"]
rob = GLOB.set_index("test").loc["1 an, 2021->2022 (robustesse)", "lift_moyen_pondere"]
h2 = GLOB.set_index("test").loc["2 ans, 2021 -> conversion 2022->2023", "lift_moyen_pondere"]
CRIT_SIGNAL = bool((r1 >= 2).all() and rob >= 2)
print("1. Mesure fiable :", "OK" if CRIT_MESURE else "ÉCHEC")
print("2. Lift 1 an par département :", r1.to_dict(), "| robustesse 2021->2022 :", rob, "->",
      "OK" if CRIT_SIGNAL else "ÉCHEC")
print("3. Horizon 2 ans (opérationnel) : lift", h2, "->", "carte de RISQUE" if h2 >= 2 else "carte de VIGILANCE descriptive")
print("4. Prairies anciennes :",
      RES[(RES.test == "1 an, 2022->2023") & (RES.modele == "régional (dép. exclu)")]
      .set_index("dep")[["lift_anciennes", "lift_anciennes_bas", "lift_anciennes_haut"]].to_dict("index"))
print("\nVERDICT :",
      "GO (observatoire + prédiction)" if CRIT_MESURE and CRIT_SIGNAL else
      "GO OBSERVATOIRE, prédiction limitée" if CRIT_MESURE else
      "NO-GO : la mesure elle-même n'est pas fiable")